1. Imports

In [11]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

2. Load Feature Dataset

In [12]:
df = pd.read_csv("../data/processed/features.csv")
df["date"] = pd.to_datetime(df["date"])

print("Shape:", df.shape)
df.head()

Shape: (1654, 14)


,date,sales,year,month,day,dayofweek,weekofyear,is_weekend,lag_1,lag_7,lag_30,rolling_7_mean,rolling_30_mean,rolling_7_std
0,2013-01-31,271254.217996,2013,1,31,3,5,0,281061.127052,247245.690995,2511.618999,319499.837745,344170.437264,70773.071168
1,2013-02-01,369402.055266,2013,2,1,4,5,0,271254.217996,290022.771930,496092.417944,330839.735365,339947.425174,71617.207631
2,2013-02-02,518887.462705,2013,2,2,5,5,1,369402.055266,413799.767975,361461.231124,345852.263183,345194.966227,98044.288785
3,2013-02-03,486336.820180,2013,2,3,6,5,1,518887.462705,430411.991233,354459.677093,353841.524461,349590.870997,107869.412143
4,2013-02-04,344308.715017,2013,2,4,0,6,0,486336.820180,285460.169953,477350.121229,362248.459470,345156.157456,103870.904953


3. Time-Based Train/Test Split (80/20)

In [13]:
split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train date range:", train["date"].min(), "to", train["date"].max())
print("Test date range:", test["date"].min(), "to", test["date"].max())

Train shape: (1323, 14)
Test shape: (331, 14)
Train date range: 2013-01-31 00:00:00 to 2016-09-17 00:00:00
Test date range: 2016-09-18 00:00:00 to 2017-08-15 00:00:00


4. Define X and y

In [14]:
FEATURES = [c for c in df.columns if c not in ["date", "sales"]]

X_train = train[FEATURES]
y_train = train["sales"]

X_test = test[FEATURES]
y_test = test["sales"]

print("Feature columns:", FEATURES)

Feature columns: ['year', 'month', 'day', 'dayofweek', 'weekofyear', 'is_weekend', 'lag_1', 'lag_7', 'lag_30', 'rolling_7_mean', 'rolling_30_mean', 'rolling_7_std']


5. Train Models

In [15]:
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))

print("Linear Regression  — MAE: {:.2f}, RMSE: {:.2f}".format(lr_mae, lr_rmse))

Linear Regression  — MAE: 82166.35, RMSE: 118465.50


In [16]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

print("Random Forest      — MAE: {:.2f}, RMSE: {:.2f}".format(rf_mae, rf_rmse))

Random Forest      — MAE: 74336.03, RMSE: 118337.50


In [17]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)
xgb.fit(X_train, y_train)
xgb_preds = xgb.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))

print("XGBoost            — MAE: {:.2f}, RMSE: {:.2f}".format(xgb_mae, xgb_rmse))

XGBoost            — MAE: 69480.42, RMSE: 109974.27


6. Compare and Select Best Model

In [18]:
results = {
    "LinearRegression": {"model": lr, "mae": lr_mae, "rmse": lr_rmse},
    "RandomForest": {"model": rf, "mae": rf_mae, "rmse": rf_rmse},
    "XGBoost": {"model": xgb, "mae": xgb_mae, "rmse": xgb_rmse},
}

best_name = min(results, key=lambda k: results[k]["rmse"])
best_model = results[best_name]["model"]

print("\n=== Model Comparison ===")
for name, r in results.items():
    tag = " ← BEST" if name == best_name else ""
    print(f"  {name:20s}  MAE={r['mae']:.2f}  RMSE={r['rmse']:.2f}{tag}")


=== Model Comparison ===
  LinearRegression      MAE=82166.35  RMSE=118465.50
  RandomForest          MAE=74336.03  RMSE=118337.50
  XGBoost               MAE=69480.42  RMSE=109974.27 ← BEST


7. Save Best Model

In [19]:
os.makedirs("../models", exist_ok=True)

joblib.dump(best_model, "../models/best_model.pkl")
print(f"Saved best model ({best_name}) to models/best_model.pkl")

Saved best model (XGBoost) to models/best_model.pkl


## 📌 Summary

- Loaded feature-engineered dataset from `data/processed/features.csv`
- Performed time-aware 80/20 train/test split
- Trained Linear Regression, Random Forest, and XGBoost models
- Evaluated all models using MAE and RMSE
- Dynamically selected best model by lowest RMSE
- Saved best model to `models/best_model.pkl`

### 🎯 Purpose
Build, compare, and persist the best-performing forecasting model.